# PRJNA391149 16S SPECTRA workflow

This notebook uses a relative-abundance matrix with samples in rows and features in columns. Feature names must match the MPA annotations used by SPECTRA-16S. The released model performs preprocessing internally and returns SPECTRA probabilities for evaluation.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings(
    "ignore",
    message=r"urllib3 .* doesn't match a supported version!",
)

from sklearn.metrics import roc_auc_score

WORK_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    path
    for path in (WORK_DIR, *WORK_DIR.parents)
    if (path / "SPECTRA_GitHub_resource").is_dir()
)
RESOURCE_ROOT = PROJECT_ROOT / "SPECTRA_GitHub_resource"
sys.path.insert(0, str(RESOURCE_ROOT / "scripts"))

from utils import predict_from_abundance_to_phenotype

ABUNDANCE_FILE = WORK_DIR / "4. Name-converted relative abundance.csv"
METADATA_FILE = WORK_DIR / "0. metadata.csv"
MRI_FILE = WORK_DIR / "5. MRI scores.csv"
PROBABILITY_FILE = WORK_DIR / "6. Probability.csv"

MRI_MODEL = RESOURCE_ROOT / "models/extensions/16S/mri_models_all.pkl"
SPECTRA_MODEL = RESOURCE_ROOT / 'models/extensions/16S/spectra_16s_model.pkl'

# 1. Relative-abundance input

In [2]:
abundance = pd.read_csv(ABUNDANCE_FILE, index_col=0, float_precision="round_trip")
abundance.index = abundance.index.astype(str)

display(pd.DataFrame({
    "value": [abundance.shape[0], abundance.shape[1]],
}, index=["samples", "observed mapped features"]))
display(abundance.head())


,value
samples,163
observed mapped features,188


,821_Bacteroides vulgatus,Unmatched_taxon_1,Unmatched_taxon_2,Unmatched_taxon_3,Unmatched_taxon_4,Unmatched_taxon_5,246787_Bacteroides cellulosilyticus,165179_Prevotella copri,Unmatched_taxon_6,Unmatched_taxon_7,...,Unmatched_taxon_152,Unmatched_taxon_153,Unmatched_taxon_154,Unmatched_taxon_155,Unmatched_taxon_156,Unmatched_taxon_157,Unmatched_taxon_158,Unmatched_taxon_159,1623_Lactobacillus ruminis,Unmatched_taxon_160
sample_id,,,,,,,,,,,,,,,,,,,,,
SRR8417365,0.545455,0.136364,0.045455,0.090909,0.090909,0.045455,0.045455,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SRR8417366,0.105263,0.052632,0.000000,0.052632,0.157895,0.000000,0.000000,0.421053,0.052632,0.105263,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SRR8417368,0.722639,0.000000,0.000000,0.000000,0.010495,0.005997,0.005997,0.000000,0.038981,0.035982,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SRR8417369,0.142857,0.000000,0.023810,0.000000,0.000000,0.023810,0.000000,0.047619,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SRR8417371,0.220820,0.000000,0.003155,0.000000,0.000000,0.170347,0.000000,0.018927,0.018927,0.157729,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# 2. SPECTRA prediction

In [3]:
result = predict_from_abundance_to_phenotype(
    Abundance=abundance,
    MRI_model_path=MRI_MODEL,
    SPECTRA_model_path=SPECTRA_MODEL,
)

mri = result["MRI"]
probability = result["probability"]
mri.to_csv(MRI_FILE)
probability.to_csv(PROBABILITY_FILE)

display(mri.head())
display(probability.head())

,control,Intestinal Diseases,Lung Diseases,IBS,Thyroid Diseases,ColorectalLesions
sample_id,,,,,,
SRR8417365,0.550,0.322,0.508,0.324,0.436,0.652
SRR8417366,0.568,0.344,0.504,0.326,0.446,0.636
SRR8417368,0.546,0.320,0.480,0.326,0.440,0.550
SRR8417369,0.544,0.350,0.480,0.326,0.412,0.574
SRR8417371,0.576,0.328,0.480,0.416,0.428,0.550


,control,Intestinal Diseases,Lung Diseases,IBS,Thyroid Diseases,ColorectalLesions
sample_id,,,,,,
SRR8417365,0.238941,0.007913,0.176069,0.190737,0.194169,0.192172
SRR8417366,0.239770,0.007909,0.175983,0.190644,0.193616,0.192078
SRR8417368,0.235471,0.007939,0.176660,0.191376,0.195737,0.192816
SRR8417369,0.235471,0.007939,0.176660,0.191376,0.195737,0.192816
SRR8417371,0.239770,0.007909,0.175983,0.190644,0.193616,0.192078


# 3. Final SPECTRA evaluation

In [4]:
metadata = pd.read_csv(METADATA_FILE).set_index("sample_id")
metadata.index = metadata.index.astype(str)
metadata = metadata.loc[probability.index]

if set(metadata["true_label"]) != {"IBS", "control"}:
    raise ValueError("Expected IBS and control labels.")

y_binary = metadata["true_label"].eq("IBS").astype(int)
auc = roc_auc_score(y_binary, probability["IBS"])

order = np.argsort(-probability.to_numpy(), axis=1)
ranked = probability.columns.to_numpy()[order]
truth = metadata["true_label"].to_numpy()


auc_table = pd.DataFrame({"AUC": [auc]}, index=["SPECTRA IBS probability"])
display(auc_table)


,AUC
SPECTRA IBS probability,0.377303
